<!--
SPDX-License-Identifier: Apache-2.0
SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
-->

# Nemotron 3.5 Lightning on NVIDIA DGX Spark

### A private, OpenAI-compatible endpoint on the box on your desk

This notebook brings up an open-weights 30B model on a single DGX Spark and drives it
from the standard OpenAI Python SDK. **Nothing leaves the machine** — no API key, no
egress, no rate limits.

---

**Everything here is open:**

| Component | License |
|---|---|
| [vLLM](https://github.com/vllm-project/vllm) | Apache 2.0 |
| [Nemotron 3.5 Lightning 30B-A3B NVFP4](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4) | OpenMDW-1.1 — weights **and training data** released |
| This sample | Apache 2.0 |

**To run it yourself on your own Spark:**

```bash
git clone https://github.com/NVIDIA/nvidia-oci-samples.git
cd nvidia-oci-samples/dgx-spark-samples/nemotron-lightning-vllm-endpoint
./setup.sh      # once
./serve.sh      # leave running
jupyter lab demo.ipynb
```

---
## 1. What is this machine?

Before anything else — confirm what we're running on.

In [ ]:
import subprocess, platform, shutil, textwrap


def unified_memory_gib():
    """Total and available memory in GiB, read from the kernel.

    nvidia-smi reports [N/A] for memory.total on GB10, and that is not a
    quirk to work around -- it is the machine stating what "unified" means.
    There is no discrete GPU memory pool to report: the CPU and the GPU share
    one allocation, so the kernel's own accounting is the only source that is
    correct here.
    """
    info = {}
    with open("/proc/meminfo") as f:
        for line in f:
            key, _, val = line.partition(":")
            info[key] = int(val.split()[0])          # /proc reports KiB
    return info["MemTotal"] / 1048576, info["MemAvailable"] / 1048576

print("=" * 66)
print("  HOST".ljust(66))
print("=" * 66)
print(f"  hostname      {platform.node()}")
print(f"  architecture  {platform.machine()}")
print(f"  python        {platform.python_version()}")

if shutil.which("nvidia-smi"):
    q = "name,memory.total,driver_version,compute_cap"
    out = subprocess.run(
        ["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip()
    for line in out.splitlines():
        name, mem, drv, cc = [p.strip() for p in line.split(",")]
        print(f"  gpu           {name}")
        if mem.startswith("[N/A"):
            total, avail = unified_memory_gib()
            print(f"  memory        {total:,.0f} GiB unified "
                  f"({avail:,.0f} GiB available right now)")
            print( "                nvidia-smi says [N/A] — there is no "
                   "separate GPU pool to report")
        else:
            print(f"  memory        {mem}")
        print(f"  driver        {drv}")
        print(f"  compute cap   {cc}   (GB10 = sm_121)")
else:
    print("  gpu           nvidia-smi not found")

print("=" * 66)
print(textwrap.dedent("""
  128 GB of unified CPU+GPU memory at 273 GB/s.
  Memory-rich, bandwidth-bound. That shapes everything below.
"""))

---
## 2. Is the endpoint up?

`serve.sh` should already be running in another terminal. Cold start is about 4 minutes,
so start it *before* you need it.

In [ ]:
import requests

BASE_URL = "http://localhost:8000/v1"
MODEL_ID = None

try:
    r = requests.get(f"{BASE_URL}/models", timeout=5)
    r.raise_for_status()
    models = r.json()["data"]
    MODEL_ID = models[0]["id"]
    print("  ENDPOINT IS UP\n")
    print(f"  url     {BASE_URL}")
    for m in models:
        print(f"  model   {m['id']}")
    print(f"  context {models[0].get('max_model_len', 'n/a'):,} tokens"
          if isinstance(models[0].get("max_model_len"), int) else "")
except Exception as e:
    print("  ENDPOINT IS NOT REACHABLE\n")
    print(f"  {type(e).__name__}: {e}\n")
    print("  Start it in another terminal:\n")
    print("      ./serve.sh\n")
    print("  Cold start takes about 4 minutes. Watch for:")
    print('      "Application startup complete"')

---
## 3. The only line that changes

Here is the whole adoption story for anyone already using a hosted model.

```python
client = OpenAI(
    base_url="https://api.openai.com/v1",     # before
    api_key=os.environ["OPENAI_API_KEY"],
)
```

```python
client = OpenAI(
    base_url="http://localhost:8000/v1",      # after — your Spark
    api_key="not-needed",
)
```

Same SDK, same request shape, same response shape.

### Where you make that change in your own code

Nothing in this sample gets edited. The line that moves is in **your** application — wherever
it constructs its client. It has the same shape everywhere:

| What you use | What you change |
| --- | --- |
| **Nothing — environment only** | `export OPENAI_BASE_URL=http://<spark>:8000/v1` and `OPENAI_API_KEY=not-needed`. The Python and Node SDKs both read these, so an app that never hard-coded the URL moves with **zero** code changes. |
| OpenAI Python SDK | `OpenAI(base_url=..., api_key="not-needed")` |
| OpenAI Node SDK | `new OpenAI({ baseURL: "...", apiKey: "not-needed" })` |
| LangChain | `ChatOpenAI(base_url=..., api_key="not-needed", model=...)` |
| LlamaIndex | `OpenAILike(api_base=..., api_key="not-needed", is_chat_model=True)` |
| Continue, Cursor, Zed | An `apiBase` field in the editor's model config |
| `curl`, or your own HTTP client | Swap the host. The paths — `/v1/chat/completions`, `/v1/models` — are identical |

Use the machine's hostname rather than `localhost` when the app runs somewhere else:
`http://spark-c251.local:8000/v1`. The `.local` matters.

**That is the port.** Not a rewrite, not an SDK swap, not a new abstraction to learn — one URL,
and an API key you no longer need because it is your hardware.


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=BASE_URL,        # <-- the only line that changes
    api_key="not-needed",     #     no key: it is your hardware
)

resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user",
               "content": "In two sentences: why would a company run a model "
                          "on hardware they own instead of a hosted API?"}],
    max_tokens=512,
    temperature=0.3,
    # Nemotron 3.5 Lightning is a reasoning model and its chat template turns
    # thinking ON by default. Reasoning and answer share one max_tokens budget,
    # so a short budget gets spent thinking and `content` comes back None.
    # Here we want a direct answer, so we switch thinking off for this request.
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

choice = resp.choices[0]
msg = choice.message

if msg.content:
    print(msg.content)
else:
    print("No answer was produced.")
    print(f"finish_reason = {choice.finish_reason!r}")
    if choice.finish_reason == "length":
        print("The budget ran out before the model reached its answer. Either raise")
        print("max_tokens, or set enable_thinking False as this cell does.")

print()
print("-" * 66)
print(f"  served by     {resp.model}")
print(f"  tokens        {resp.usage.prompt_tokens} in / "
      f"{resp.usage.completion_tokens} out   [finish: {choice.finish_reason}]")


---
## 4. Streaming, with the numbers visible

Time-to-first-token and decode rate, measured live. Real hardware has jitter —
you can see it here.

In [ ]:
import time, sys

def _text(obj, *fields):
    """First non-empty string among the named attributes."""
    for f in fields:
        v = getattr(obj, f, None)
        if isinstance(v, str) and v:
            return v
    return None


def ask(prompt, system=None, max_tokens=1536, temperature=0.3,
        thinking=True, show_thinking=True, show_stats=True):
    """Stream an answer from the Spark and report TTFT and decode rate.

    Two things this handles that a plain OpenAI streaming loop does not:

    1. The reasoning trace arrives on its own field, because vLLM is serving
       with --reasoning-parser nemotron_v3. It is `reasoning` on vLLM 0.26 and
       later, `reasoning_content` before that, and never `content`. Watch only
       `content` and a thinking model looks like it produced nothing.
    2. Reasoning and answer share one max_tokens budget. Set it too low and the
       budget is spent thinking, `content` stays empty and finish_reason is
       'length' -- which is easy to misread as the model having nothing to say.

    thinking=False switches reasoning off for the request, which the model's
    chat template supports directly. Faster, and right when you want an answer
    rather than a demonstration of how it got there.
    """
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    t0 = time.perf_counter()
    ttft = None
    n_think = n_answer = 0
    chunks = []
    in_think = False
    finish = None

    stream = client.chat.completions.create(
        model=MODEL_ID, messages=messages,
        max_tokens=max_tokens, temperature=temperature, stream=True,
        extra_body={"chat_template_kwargs": {"enable_thinking": bool(thinking)}},
    )

    for chunk in stream:
        if not chunk.choices:
            continue
        choice = chunk.choices[0]
        finish = choice.finish_reason or finish
        think = _text(choice.delta, "reasoning", "reasoning_content")
        piece = _text(choice.delta, "content")

        if think:
            if ttft is None:
                ttft = time.perf_counter() - t0
            n_think += 1
            if show_thinking:
                if not in_think:
                    sys.stdout.write("[thinking] ")
                    in_think = True
                sys.stdout.write(think)
                sys.stdout.flush()

        if piece:
            if ttft is None:
                ttft = time.perf_counter() - t0
            if in_think:
                sys.stdout.write("\n\n[answer]\n")
                in_think = False
            n_answer += 1
            chunks.append(piece)
            sys.stdout.write(piece)
            sys.stdout.flush()

    total = time.perf_counter() - t0
    n = n_think + n_answer

    if finish == "length" and not chunks:
        print(f"\n\n  Budget exhausted after {n_think} reasoning tokens, with no answer.")
        print(f"  Raise max_tokens above {max_tokens}, or pass thinking=False.")

    if show_stats:
        decode = (n - 1) / (total - ttft) if ttft and total > ttft else 0.0
        head = f"{ttft*1000:>7.1f} ms" if ttft else "    n/a "
        print("\n" + "-" * 66)
        print(f"  TTFT {head}   decode {decode:>5.1f} tok/s   total {total:>5.2f} s")
        detail = f"{n} tokens"
        if n_think:
            detail += f"  ({n_think} reasoning + {n_answer} answer)"
        if finish and finish != "stop":
            detail += f"   [finish: {finish}]"
        print(f"  {detail}")
    return "".join(chunks)


_ = ask("Explain what 'unified memory' means on a Grace Blackwell system, "
        "and why it matters for running large models.",
        max_tokens=1536)


### 4b. Latency, measured rather than quoted

Two numbers people always ask for, taken from this machine at batch 1. **Time to first token**
is how long you wait before anything appears; **decode rate** is how fast it writes after that.

Run it against a short prompt and against a long one. Prefill scales with input length and decode
does not, so the interesting result is how little the second number moves.

No API key and no network — this measures the endpoint on your own desk. There is deliberately
no hosted comparison here: the free public gateways queue requests, so timing them measures the
queue rather than the model, and reporting that as latency would be dishonest in both directions.


In [ ]:
import time

# Batch 1, so this is latency rather than throughput. One GB10 has ~273 GB/s of
# memory bandwidth and decode is bandwidth-bound -- which is why concurrent
# requests contend -- but a single stream is what an interactive tool actually
# feels like, and it is the honest number for a developer endpoint.
#
# thinking=False keeps the measurement about the serving path. With reasoning on
# you would mostly be timing how long the model chose to think, which varies by
# prompt and says nothing about the machine.
#
# Both requests ask for the same answer and the same output length. Only the
# input length differs, so the decode rates are comparable and the change in
# TTFT is prefill and nothing else.

TASK = "In about 120 words, explain why memory bandwidth matters when serving a language model."
FILLER = "The endpoint runs locally on a DGX Spark, and nothing leaves the box. "


def measure(prompt, label, max_tokens=256):
    t0 = time.perf_counter()
    ttft, n = None, 0
    stream = client.chat.completions.create(
        model=MODEL_ID, messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens, temperature=0.0, stream=True,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    for chunk in stream:
        if not chunk.choices:
            continue
        if getattr(chunk.choices[0].delta, "content", None):
            if ttft is None:
                ttft = time.perf_counter() - t0
            n += 1
    total = time.perf_counter() - t0
    decode = (n - 1) / (total - ttft) if ttft and n > 1 and total > ttft else 0.0
    # ~4 chars per token is a rough count, and labelled as rough. Reporting it as
    # exact would be inventing precision the client never received.
    return {"label": label, "in_tok": len(prompt) // 4, "out_tok": n,
            "ttft_ms": (ttft or 0) * 1000, "decode": decode}


print("  warming up...", flush=True)
measure(TASK, "warmup", max_tokens=16)     # the first call pays one-off setup costs

rows = [measure(TASK, "short prompt"),
        measure(FILLER * 460 + "\n\n" + TASK, "~8k prompt")]

print(f"\n  {'prompt':<15}{'input':>12}{'output':>10}{'TTFT':>12}{'decode':>13}")
print("  " + "-" * 62)
for r in rows:
    print(f"  {r['label']:<15}{r['in_tok']:>8,} tok{r['out_tok']:>7} tok"
          f"{r['ttft_ms']:>9.1f} ms{r['decode']:>9.1f} tok/s")

dt = rows[1]["ttft_ms"] - rows[0]["ttft_ms"]
dd = rows[1]["decode"] - rows[0]["decode"]
print(f"\n  Roughly {rows[1]['in_tok'] - rows[0]['in_tok']:,} more input tokens cost "
      f"{dt:+.0f} ms of prefill, and moved the")
print(f"  decode rate by {dd:+.1f} tok/s. Long prompts are cheap on this machine;")
print( "  generation is the bottleneck, and generation is bound by memory bandwidth.")


### 4c. The same question, asked of every endpoint

§4b measured this machine. This asks how it compares — **time to first token** and **decode
rate**, one request at a time, against the local Spark and four hosted models.

Those two metrics are chosen because they compare fairly and total wall time does not. Total
depends on how many tokens a model decided to emit, and a reasoning model that thinks for 900
tokens is not slower than one that answers in 40 — it did more work. TTFT is independent of
output length; decode rate is per-token by construction.

**Read the whiskers, not just the bars.** For a hosted endpoint this is end-to-end from wherever
you are sitting: your network path, the gateway's routing, queueing behind other tenants, and
only then the model. Those cannot be separated from outside, so a long whisker means *this
endpoint is queueing*, not *this model is slow*. The pair worth trusting is the same model in
two places, because everything except the serving path is held constant — read the rest as
context.

Results load from `results/latency.json`. Set `LATENCY_FORCE_RERUN = True` to measure again;
that needs `NVIDIA_API_KEY` for the hosted endpoints and takes a few minutes. The local endpoint
alone needs no key at all:

```bash
./latency.py --only spark
```


In [ ]:
# ---------------------------------------------------------------------------
# Same pattern as the benchmark below: read the saved result by default, and
# re-measure only when explicitly asked. A shared gateway is a poor thing to
# depend on live, and a probe that takes minutes is a poor thing to run in
# front of an audience.
#
# Latency is a snapshot in a way accuracy is not. A gateway at 18:00 is not the
# same gateway at 09:00, so the file records when it was taken and the cell
# prints it. If that timestamp looks stale for the claim you are about to make,
# re-run rather than quoting it.
# ---------------------------------------------------------------------------

LATENCY_FORCE_RERUN = False      # True = measure now. Needs NVIDIA_API_KEY. Minutes.
LATENCY_SAMPLES     = 5          # sequential samples per endpoint, one warmup discarded

import json, pathlib, subprocess, sys
from IPython.display import Image, display

LATENCY_PATH = pathlib.Path("results/latency.json")

if LATENCY_FORCE_RERUN:
    cmd = [sys.executable, "latency.py", "--samples", str(LATENCY_SAMPLES)]
    print("  " + " ".join(cmd) + "\n")
    subprocess.run(cmd, check=True)
    subprocess.run([sys.executable, "make_latency_chart.py"], check=True)

if not LATENCY_PATH.exists():
    print("  No results/latency.json yet. Run ./latency.py --only spark for the local")
    print("  endpoint (no key needed), or set LATENCY_FORCE_RERUN = True above.")
else:
    lat = json.loads(LATENCY_PATH.read_text())
    ok  = [r for r in lat["endpoints"] if not r.get("skipped")]
    ok.sort(key=lambda r: (0 if "local" in (r.get("location") or "").lower() else 1,
                           r["ttft_ms"]["median"]))

    print(f"  measured   {lat['generated']}")
    print(f"  method     {lat['samples_requested']} sequential samples, concurrency 1, "
          f"max_tokens {lat['max_tokens']}\n")
    print(f"  {'endpoint':<20}{'TTFT median':>14}{'range':>20}{'decode':>16}")
    print("  " + "-" * 70)
    for r in ok:
        t, d = r["ttft_ms"], r["decode_tok_s"]
        star = " *" if "local" in (r.get("location") or "").lower() else "  "
        rng = f"{t['min']:.0f}–{t['max']:.0f} ms"
        approx = "" if r.get("decode_exact", True) else " ~"
        print(f"  {r['name']:<18}{star}{t['median']:>11.0f} ms{rng:>20}"
              f"{d['median']:>11.1f} tok/s{approx}")
    for r in lat["endpoints"]:
        if r.get("skipped"):
            print(f"  {r['name']:<20}  skipped — {r['skipped'][:44]}")
    print("\n  * running locally on this DGX Spark")

    # One chunk is one token on local vLLM, so counting chunks is right there and
    # wrong against a gateway that packs several per chunk -- and it fails upward.
    if any(not r.get("decode_exact", True) for r in ok):
        print("\n  ~ decode rate inferred from stream chunks, because that endpoint returned")
        print("    no token count. A gateway that batches chunks, or flushes a burst at the")
        print("    end, inflates this figure. Do not quote it.")
        print("    TTFT is unaffected: it is the time to the first chunk either way.")

    # A wide spread is the finding, not a blemish on it -- say so rather than
    # letting a reader take the median at face value.
    noisy = [r for r in ok if r["ttft_ms"]["max"] > 3 * max(r["ttft_ms"]["median"], 1)]
    if noisy:
        print("\n  Wide spread on: " + ", ".join(r["name"] for r in noisy))
        print("  Worst case is more than 3x the median, which is queueing rather than")
        print("  model speed. Quote the range for these, not the median.")

    png = pathlib.Path("results/latency.png")
    if png.exists():
        display(Image(filename=str(png)))


---
## 5. Your turn

Paste a prompt below and run the cell. This is going to a 30B model on one
desk-side machine, and nothing about it touches the internet.

In [ ]:
# ---------------------------------------------------------------------------
# Replace the text below and re-run.
#
# thinking=False keeps it snappy for a live audience. Drop the argument to
# watch the reasoning trace stream in first.
#
# On the default prompt: asking an open question about local-versus-hosted
# invites "bypassing content filters" as an answer, because it is a real answer.
# True, and not what you want on a shared screen in front of customers. Naming
# *enterprise workloads*, and asking for the counter-case too, keeps it on the
# ground and reads as more credible than a one-sided pitch.
# ---------------------------------------------------------------------------

PROMPT = ("Name three enterprise workloads where running a model on your own hardware "
          "is the right call, and one where a hosted API is the better choice. "
          "Two sentences each.")

_ = ask(PROMPT, max_tokens=768, thinking=False)


---
## 6. It is an agent, not a chatbot

The model is served with structured tool calling enabled:

```
--enable-auto-tool-choice --tool-call-parser qwen3_coder
```

so tool calls come back as structured `tool_calls`, not text to scrape.

Below we give it two unrelated tools and a task that needs both. Nobody tells it
to chain them — it works out that it has to search first, then convert.

The tool implementations are plain local Python (`demo_tools.py`). Swap them for
Fusion APIs, an internal service, or a database and the loop is unchanged.

In [ ]:
import json
from demo_tools import TOOL_SCHEMAS, TOOL_IMPLEMENTATIONS

for t in TOOL_SCHEMAS:
    fn = t["function"]
    params = ", ".join(fn["parameters"]["properties"])
    print(f"  {fn['name']}({params})")
    print(f"      {fn['description'][:80]}")
print(f"\n  {len(TOOL_SCHEMAS)} tools available. Real functions, executed locally.")

In [ ]:
def run_agent(task, max_rounds=6, verbose=True):
    """Full tool-calling loop: model decides, we execute, model continues."""
    messages = [{"role": "user", "content": task}]
    calls_made = []

    for round_no in range(1, max_rounds + 1):
        resp = client.chat.completions.create(
            model=MODEL_ID, messages=messages,
            tools=TOOL_SCHEMAS, tool_choice="auto",
            max_tokens=2048, temperature=0.0,
        )
        msg = resp.choices[0].message

        # vLLM >= 0.26 exposes the reasoning trace as `reasoning`,
        # older builds used `reasoning_content`. Handle both.
        reasoning = getattr(msg, "reasoning", None) or \
                    getattr(msg, "reasoning_content", None)
        if reasoning and verbose:
            snippet = reasoning.strip().replace("\n", " ")[:150]
            print(f"  [round {round_no}] thinking: {snippet}...")

        if not msg.tool_calls:
            if verbose:
                print(f"\n  [round {round_no}] final answer\n")
            print("  " + "\n  ".join(msg.content.strip().split("\n")))
            return {"answer": msg.content, "calls": calls_made, "rounds": round_no}

        messages.append(msg)
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            calls_made.append(name)
            if verbose:
                print(f"  [round {round_no}] CALL  {name}({json.dumps(args)})")

            impl = TOOL_IMPLEMENTATIONS.get(name)
            result = impl(**args) if impl else {"error": f"unknown tool {name}"}

            if verbose:
                print(f"  [round {round_no}] ->    {json.dumps(result)[:110]}")

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result),
            })

    return {"answer": None, "calls": calls_made, "rounds": max_rounds}


TASK = ("Find the cheapest flight from SFO to AUS on 2026-09-14, "
        "then tell me that fare in euros.")

print(f"  TASK: {TASK}\n")
print("=" * 66)
result = run_agent(TASK)
print("=" * 66)
print(f"\n  {len(result['calls'])} tool calls across {result['rounds']} rounds: "
      f"{' -> '.join(result['calls'])}")

---
## 7. The harder question: does it know when *not* to call?

The agent loop above is an anecdote. This is the evidence.

Most function-calling benchmarks score whether a call was *correct* once the model
decided to make one. [`nvidia/When2Call`](https://huggingface.co/datasets/nvidia/When2Call)
(CC-BY-4.0) scores something more useful: **whether it should have called anything at all.**

| Label | Correct behaviour |
|---|---|
| `tool_call` | Call the tool — every required argument is available |
| `request_for_info` | Ask a follow-up — a required argument was deliberately withheld |
| `cannot_answer` | Decline — no available tool covers the request |

Scoring is deterministic — we observe what the server did, no LLM judge — so you can
reproduce these numbers on your own hardware. Four metrics:

- **Decision accuracy** — did it call exactly when it should have?
- **Actionable decision accuracy** — correct decision *and* it either called a tool or
  produced real text. Stops a model scoring well by staying silent.
- **Tool-selection accuracy** — when it correctly called, was it the right tool?
- **Over-call rate** — how often it fired a tool it shouldn't have. **Lower is better.
  This is the number that predicts agent misbehaviour in production.**

### 7a. Five models, same 120 examples

Read from `results/summary.json`, which ships with this repo. Identical prompts and identical
converted tool schemas for every model.

**Run the benchmark yourself.** Nothing here is precomputed magic — the file above is the output
of a script in this directory, and you can regenerate it on your own hardware:

1. Get a free NVIDIA API key. Create an account at [build.nvidia.com](https://build.nvidia.com/),
   then generate an **NGC Personal API Key** with the *NVIDIA Public API Endpoints* service
   enabled. Anyone can do this — no NVIDIA affiliation required, and the free tier is enough.
2. Export it, without putting it in your shell history:
   ```bash
   read -rs -p "NVIDIA_API_KEY: " NVIDIA_API_KEY && export NVIDIA_API_KEY && echo
   ```
3. Check every endpoint before committing time to it:
   ```bash
   ./preflight.py --real 2
   ```
4. Run it:
   ```bash
   ./run_benchmark.py --config endpoints.json --n 40 --save-raw
   python3 make_chart.py
   ```

Budget a few hours. Most of that is waiting on shared free-tier gateways rather than model
compute, so `--only spark` gives you the local result in about twelve minutes if you only want
the model running on your own box — and that one needs no key at all.

Setting `FORCE_RERUN = True` in the cell below does the same thing from inside this notebook.


In [ ]:
# ---------------------------------------------------------------------------
# Results come from a file by default. Set FORCE_RERUN = True to measure again.
#
# The sweep is 600 requests against five endpoints and takes a few hours, most
# of it waiting on shared free-tier gateways. That is fine unattended and wrong
# in front of an audience, so this cell reads results/summary.json unless you
# explicitly ask it not to.
#
# To produce that file yourself, see "Run the benchmark yourself" below.
# ---------------------------------------------------------------------------

FORCE_RERUN  = False        # True = measure now. Needs NVIDIA_API_KEY. Hours.
N_PER_LABEL  = 40           # 40 per label x 3 labels = 120 examples per model

import json, os, pathlib, subprocess, sys, datetime

SUMMARY_PATH = pathlib.Path("results/summary.json")


def _describe(d, source):
    print(f"  source     {source}")
    print(f"  generated  {d.get('generated', 'unknown')}")
    ds = d.get("dataset") or {}
    if ds:
        print(f"  dataset    {ds.get('repo')} config={ds.get('config')!r} "
              f"split={ds.get('split')!r}")
    print(f"  examples   {d.get('n_examples')} per model")
    print(f"  scored     {len(d.get('models', []))} models, "
          f"{len(d.get('failed', []))} produced no data")
    for f in d.get("failed", []):
        print(f"             {f['name']}: {f['reason'][:60]}")


if FORCE_RERUN:
    specs = json.loads(pathlib.Path("endpoints.json").read_text())["endpoints"]
    missing = sorted({s["api_key_env"] for s in specs
                      if s.get("api_key_env") and not os.environ.get(s["api_key_env"])})
    if missing:
        print(f"  {', '.join(missing)} is not set in this kernel's environment.\n")
        print("  Jupyter inherits the environment of the shell that launched it, so")
        print("  exporting the key in another terminal will not reach this kernel.\n")
        print("  Stop Jupyter, export the key, and start it again:\n")
        print('      read -rs -p "NVIDIA_API_KEY: " NVIDIA_API_KEY && export NVIDIA_API_KEY')
        print("      jupyter lab --no-browser --ip 127.0.0.1 --port 8888\n")
        print("  Or run the sweep from a terminal and re-run this cell with")
        print("  FORCE_RERUN = False to pick up the file it writes.")
        raise RuntimeError(f"{', '.join(missing)} not set")

    print("  Re-running the full sweep. This takes hours, not minutes.\n")
    import run_benchmark as rb
    SUMMARY = rb.sweep(n_per_label=N_PER_LABEL, save_raw=True)
    subprocess.run([sys.executable, "make_chart.py"], check=True)
    subprocess.run([sys.executable, "make_chart.py", "--compact",
                    "--out", "results/when2call-slide.png"], check=True)
    print()
    _describe(SUMMARY, "measured just now")

elif SUMMARY_PATH.exists():
    SUMMARY = json.loads(SUMMARY_PATH.read_text())
    _describe(SUMMARY, SUMMARY_PATH)
    age_note = ""
    try:
        gen = datetime.datetime.strptime(SUMMARY["generated"], "%Y-%m-%d %H:%M:%S")
        days = (datetime.datetime.now() - gen).days
        if days > 30:
            age_note = (f"\n  NOTE: these results are {days} days old. Model weights, "
                        f"runtimes and\n        the dataset have all moved under this "
                        f"benchmark before -- re-run\n        before quoting them.")
    except (KeyError, ValueError):
        pass
    if age_note:
        print(age_note)

else:
    raise FileNotFoundError(
        "results/summary.json not found.\n\n"
        "Generate it with:\n"
        "    read -rs -p 'NVIDIA_API_KEY: ' NVIDIA_API_KEY && export NVIDIA_API_KEY\n"
        "    ./preflight.py --real 2\n"
        "    ./run_benchmark.py --config endpoints.json --n 40 --save-raw\n\n"
        "Or set FORCE_RERUN = True above, having launched Jupyter from a shell\n"
        "with NVIDIA_API_KEY exported.")


In [ ]:
from when2call import pct, overlaps

summary = SUMMARY          # set by the LIVE_BENCHMARK cell above
models  = sorted(summary["models"],
                 key=lambda m: -m["decision_accuracy"]["rate"])

print(f"  nvidia/When2Call · {summary['n_examples']} examples · run {summary['generated']}\n")
print(f"  {'model':<20}{'decision accuracy':<30}{'tool sel':<11}{'over-call'}")
print("  " + "-" * 73)
for m in models:
    d, t, o = (m["decision_accuracy"], m["tool_selection_accuracy"],
               m["over_call_rate"])
    star = " *" if "local" in m.get("location", "").lower() else "  "
    # Build the accuracy column as its own string before padding it.
    # Adjacent string literals are concatenated *before* the attribute access,
    # so f"a" f"b".ljust(26) pads the whole line -- which by that point is
    # already past 26 characters, making the call a silent no-op.
    acc = (f"{d['k']:>3}/{d['n']:<4}{pct(d['rate']):>6}  "
           f"[{pct(d['lo'])}–{pct(d['hi'])}]")
    print(f"  {m['name']:<18}{star}{acc:<30}"
          f"{t['k']:>2}/{t['n']:<4}   {pct(o['rate']):>6}")

for f in summary.get("failed", []):
    print(f"  {f['name']:<20}  no data — {f['reason'][:44]}")

print("\n  * running locally on this DGX Spark")

In [ ]:
from IPython.display import Image, display
display(Image(filename="results/when2call.png"))

### 7b. Reading it honestly

Two things worth saying out loud, because the chart invites a wrong reading.

In [ ]:
import when2call as w2c
# Reads the result rather than asserting it, and tests it properly.
#
# Overlapping confidence intervals are conservative -- two can overlap while the
# difference is real -- so the eyeball test alone is not enough. And with five
# models there are ten pairwise comparisons per metric, where testing each at
# 0.05 would expect about one false positive from noise. Holm-Bonferroni
# corrects for that.

dec  = w2c.holm_pairwise(models, "decision_accuracy")
over = w2c.holm_pairwise(models, "over_call_rate")
dec_sig  = [x for x in dec  if x["significant"]]
over_sig = [x for x in over if x["significant"]]

n_dec  = models[0]["decision_accuracy"]["n"]
margin = (models[0]["decision_accuracy"]["hi"]
          - models[0]["decision_accuracy"]["lo"]) / 2 * 100

print(f"  {len(models)} models, {len(dec)} pairwise comparisons per metric,")
print(f"  Holm-corrected at alpha = 0.05.\n")

if not dec_sig:
    print("  1. DECISION ACCURACY IS A GENUINE TIE.")
    print(f"     Not one of the {len(dec)} comparisons survives correction.")
    best, worst = models[0], models[-1]
    print(f"     Spread runs {pct(worst['decision_accuracy']['rate'])} to "
          f"{pct(best['decision_accuracy']['rate'])}, and the widest gap")
    print(f"     ({worst['name']} vs {best['name']}) comes to p={dec[0]['p']:.3f}, "
          f"needing p<{dec[0]['threshold']:.3f}.")
    print(f"     At n={n_dec} the interval is about +/-{margin:.1f} points. Ranking these")
    print("     models on this metric would be reading noise.\n")
else:
    print("  1. SOME DECISION-ACCURACY DIFFERENCES ARE REAL.")
    for x in dec_sig:
        print(f"     {x['a']} vs {x['b']}: p={x['p']:.4f} < {x['threshold']:.4f}")
    print()

if over_sig:
    names = {x["a"] for x in over_sig} & {x["b"] for x in over_sig}
    worst = max(models, key=lambda m: m["over_call_rate"]["rate"])
    best  = min(models, key=lambda m: m["over_call_rate"]["rate"])
    print("  2. OVER-CALL RATE IS WHERE A REAL DIFFERENCE APPEARS.")
    print(f"     {worst['name']} fires a tool it shouldn't "
          f"{pct(worst['over_call_rate']['rate'])} of the time; "
          f"{best['name']} does it {pct(best['over_call_rate']['rate'])},")
    print("     on identical examples.")
    print(f"     {len(over_sig)} of {len(over)} comparisons survive correction, and every one")
    print(f"     of them involves {worst['name']}:")
    for x in over_sig:
        other = x["b"] if x["a"] == worst["name"] else x["a"]
        print(f"       vs {other:<18} p={x['p']:.4f} < {x['threshold']:.4f}")
    print("\n     This is the metric that matters in production. An agent with write")
    print("     access that over-calls is one that makes changes nobody asked for.\n")
else:
    print("  2. OVER-CALL RATE DOES NOT SEPARATE THESE MODELS EITHER.\n")

spark = next((m for m in models if "local" in m.get("location", "").lower()), None)
host  = next((m for m in models if m["name"] == "nemotron-hosted"), None)
if spark and host:
    p = w2c.two_proportion_p(spark["decision_accuracy"]["k"], spark["decision_accuracy"]["n"],
                             host["decision_accuracy"]["k"], host["decision_accuracy"]["n"])
    print("  CONTROL: the same model, two places.")
    print(f"     local {pct(spark['decision_accuracy']['rate'])}  vs  "
          f"hosted {pct(host['decision_accuracy']['rate'])}   p={p:.2f}  ->  "
          f"{'equivalent within noise' if p > 0.05 else 'DIFFERENT -- investigate'}")
    print("     If these two disagreed, the cross-model comparison would mean nothing.")


### 7c. Run a slice live, right now

The table above took about nine minutes for the full 120 examples. Here's a small
stratified slice against the Spark so you can watch the harness actually work —
same code, same scoring, just fewer examples.

In [ ]:
import time
import when2call as w2c
from run_benchmark import call_with_backoff

N_PER_LABEL = 4          # 12 examples — about a minute. Raise for a longer run.

examples = w2c.load_examples(n_per_label=N_PER_LABEL, seed=7)
print(f"  {len(examples)} examples, stratified across the three labels\n")

records = []
t0 = time.perf_counter()
for i, ex in enumerate(examples, 1):
    msg, finish, err = call_with_backoff(
        client, model=MODEL_ID,
        messages=w2c.build_messages(ex),
        tools=w2c.to_openai_tools(ex["tools"]),
        max_tokens=3072)
    if err:
        rec = w2c.Record(label=ex["label"], called=False, tool_name=None,
                         gold_tool=ex.get("gold_tool"), has_text=False,
                         finish_reason=None, error=err)
    else:
        rec = w2c.observe(msg, finish)
        rec.label = ex["label"]
        rec.gold_tool = ex.get("gold_tool")
    records.append(rec)

    mark = "OK  " if rec.decision_correct else "MISS"
    did  = f"called {rec.tool_name}" if rec.called else "no call"
    print(f"  [{i:>2}/{len(examples)}] {mark}  want {rec.label:<17} got {did}")

print(f"\n  {time.perf_counter()-t0:.1f}s on this box\n")
print(w2c.summarise("live slice (DGX Spark)", w2c.score(records)))
print("  Small n — wide intervals. This proves the harness runs, not a ranking.")

---
## 8. What is left over

The model is resident. How much of the box is still free?

In [ ]:
import re, requests

# Everything on this slide is read from the machine you are sitting at, not
# quoted from a spec sheet. Numbers below will differ from anyone else's.

total, avail = unified_memory_gib()          # defined in the first cell
used = total - avail
bar_w = 44
filled = int(bar_w * used / total)
print("  UNIFIED MEMORY")
print("  [" + "#" * filled + "." * (bar_w - filled) + "]")
print(f"  {used:.1f} GiB in use   {avail:.1f} GiB available   "
      f"{total:.1f} GiB total\n")

# vLLM publishes its KV cache sizing on /metrics as labels on
# vllm:cache_config_info. Label names have moved between releases, so read them
# defensively and say so plainly if they are not there rather than inventing a
# number.
try:
    text = requests.get(BASE_URL.rsplit("/v1", 1)[0] + "/metrics", timeout=5).text
    line = next(l for l in text.splitlines() if l.startswith("vllm:cache_config_info"))
    labels = dict(re.findall(r'(\w+)="([^"]*)"', line))
    blocks = int(labels["num_gpu_blocks"])
    block_size = int(labels["block_size"])
    # Report the inputs, not a product presented as fact. blocks x block_size
    # came out 27% above the "GPU KV cache size" vLLM prints at startup on this
    # build -- 31.5M against 24.7M -- so at least one of these labels does not
    # mean what the arithmetic assumes. The startup log is the authoritative
    # figure; this is a cross-check, and it is currently disagreeing.
    print(f"  KV CACHE\n  {blocks:,} blocks x {block_size} per block")
    print( "  For the token capacity, read 'GPU KV cache size' in the serve.sh")
    print( "  startup log -- that is the number vLLM itself reports.")
    print(f"  cache dtype             {labels.get('cache_dtype', 'n/a')}")
    print( "\n  That cache is memory vLLM has already reserved -- not memory\n"
           "  still free. --gpu-memory-utilization sets the split; lower it\n"
           "  and the difference returns to the pool above.")
except Exception as e:
    print(f"  KV cache metrics unavailable ({type(e).__name__}). "
          f"The figure is also printed in the serve.sh log at startup.")

if MODEL_ID:
    ctx = requests.get(f"{BASE_URL}/models", timeout=5).json()["data"][0]
    if isinstance(ctx.get("max_model_len"), int):
        print(f"  serving context         {ctx['max_model_len']:,} tokens")

print("""
  30B total parameters, 3B active per token. The mixture-of-experts routing is
  why a 30B model serves comfortably from something that sits on a desk: you
  pay memory for all 30B, but compute for 3B.

  Whatever is left in that free column is headroom for the next thing -- a
  second model, a fine-tuning run, an embedding index -- on hardware already
  paid for, with no per-token cost and no rate limit.
""")


---
## 9. Where this goes next

Same box, same 128 GB — inference is the easy half.

| | |
|---|---|
| **Optimize** | [Quantize to NVFP4 with Model Optimizer](https://build.nvidia.com/spark/nvfp4-quantization) — ~3.5× memory reduction vs FP16, accuracy close to FP8 |
| **Train** | [Fine-tune with PyTorch](https://build.nvidia.com/spark/pytorch-fine-tune) — FSDP + LoRA, up to 70B across two Sparks · [Unsloth](https://build.nvidia.com/spark/unsloth) · [LLaMA-Factory](https://build.nvidia.com/spark/llama-factory) |
| **Scale** | [Connect two Sparks](https://build.nvidia.com/spark/connect-two-sparks) over 200 Gb/s for 256 GB pooled |

Fine-tuning is memory-bound. Full fine-tuning of Llama 3.2 3B, LoRA on Llama 3.1 8B,
and QLoRA on Llama 3.3 70B all run here — **and none of them fit on a 32 GB consumer GPU.**
That is the argument for this box, and it is the part you cannot do against a hosted API at all.

---

### Being straight about the limits

- **A single GB10 is a development endpoint, not a shared production one.** 273 GB/s of
  memory bandwidth means concurrent requests contend and largely serialise. One user or
  one CI job: excellent. Thirty simultaneous users: not what this is for.
- **GB10 has no native FP4 compute.** NVFP4 here is a *memory* optimisation — weights
  stored 4-bit, decompressed to compute via the Marlin kernel. A large win on a
  bandwidth-bound part, but it is not FP4 tensor-core acceleration.
- **Do not buy this for throughput.** Buy it for capacity, data residency, zero marginal
  cost per token, and no rate limits.

---

### Run it yourself

```bash
git clone https://github.com/NVIDIA/nvidia-oci-samples.git
cd nvidia-oci-samples/dgx-spark-samples/nemotron-lightning-vllm-endpoint
./setup.sh && ./serve.sh
```

Questions, issues, or a sample of your own to contribute:
[github.com/NVIDIA/nvidia-oci-samples/issues](https://github.com/NVIDIA/nvidia-oci-samples/issues)